# Quran Corpus — Cross-Lingual Sentence-Encoder Retrieval (Experiment 2)

Because every verse is a gold semantic equivalent across 40 languages, we get cross-lingual paraphrase pairs for free — including low-resource languages that Tatoeba/FLORES skip. This notebook benchmarks multilingual sentence encoders on **verse-level cross-lingual retrieval** and probes whether embeddings cluster by *meaning* or by *language family*.

**Runtime:** use a GPU runtime (Runtime -> Change runtime type -> GPU). ~10-30 min depending on models.


## 0. Install

In [ ]:
!pip -q install sentence-transformers pandas scikit-learn matplotlib 2>/dev/null
import os, glob, csv, json, io, zipfile, time, itertools
import numpy as np, pandas as pd
print('ready')


## 1. Upload data (same `translations.zip` as Experiment 1)

In [ ]:
from google.colab import files
DATA_DIR='/content/translations'; os.makedirs(DATA_DIR, exist_ok=True)
print('Upload translations.zip:')
up=files.upload()
for fn in up:
    if fn.lower().endswith('.zip'):
        zipfile.ZipFile(io.BytesIO(up[fn])).extractall(DATA_DIR)
    elif fn.lower().endswith('.csv'):
        open(os.path.join(DATA_DIR,fn),'wb').write(up[fn])
csv_files=sorted(glob.glob(os.path.join(DATA_DIR,'**','*.csv'),recursive=True))
print(len(csv_files),'files')


## 2. Parse + align (keep verses present in all files)

In [ ]:
def load_csv(fp):
    keys={}
    with open(fp,encoding='utf-8',errors='replace') as f:
        rdr=csv.reader(f); started=False
        for row in rdr:
            if not started:
                low=[(x or '').strip().lower() for x in row]
                if 'sura' in low and 'aya' in low:
                    started=True; iS=low.index('sura'); iA=low.index('aya'); iT=low.index('translation') if 'translation' in low else 3
                continue
            try: keys[(int(row[iS]),int(row[iA]))]=row[iT]
            except: pass
    return keys
def lang(fp): return os.path.basename(fp).split('_')[0]
from collections import Counter
data={fp:load_csv(fp) for fp in csv_files}
cnt=Counter(lang(f) for f in csv_files)
disp={f:(lang(f) if cnt[lang(f)]==1 else lang(f)+'-'+os.path.basename(f).split('_')[1]) for f in csv_files}
common=None
for k in data.values(): common=set(k) if common is None else common&set(k)
common=sorted(common); print('aligned verses:',len(common),'| languages:',len(data))


## 3. Choose models + a verse sample
Full 6,236-verse encoding for many models is heavy; start with a representative random sample (set `SAMPLE=None` to use all verses).

In [ ]:
SAMPLE=1000   # number of verses to evaluate on; set None for all 6236
rng=np.random.default_rng(0)
idx = common if SAMPLE is None else [common[i] for i in rng.choice(len(common), size=min(SAMPLE,len(common)), replace=False)]
print('evaluating on', len(idx), 'verses')

MODELS=[
    'sentence-transformers/LaBSE',
    'sentence-transformers/paraphrase-multilingual-mpnet-base-v2',
    'intfloat/multilingual-e5-base',
    'BAAI/bge-m3',
]
# pick a subset of languages to keep pairwise retrieval tractable (or use all)
LANGS=list(data.keys())   # all; reduce if slow
print(len(LANGS),'languages')


## 4. Cross-lingual retrieval benchmark → `retrieval_results.csv`
For each model and each ordered language pair (Lsrc -> Ltgt), encode the same verses in both languages and measure **top-1 retrieval accuracy** (is verse i in Lsrc nearest to verse i in Ltgt?) plus mean reciprocal rank. Averaged per model and per language.

In [ ]:
from sentence_transformers import SentenceTransformer
import torch
from sklearn.preprocessing import normalize

def encode(model, texts, prefix=''):
    return model.encode([prefix+t for t in texts], batch_size=64, convert_to_numpy=True, show_progress_bar=False, normalize_embeddings=True)

rows=[]
per_lang=[]
for mid in MODELS:
    try:
        model=SentenceTransformer(mid, device='cuda' if torch.cuda.is_available() else 'cpu')
    except Exception as e:
        print('skip',mid,str(e)[:80]); continue
    pre='query: ' if 'e5' in mid else ''
    emb={}
    for fp in LANGS:
        emb[fp]=encode(model,[str(data[fp][k]) for k in idx], pre)
    # all-pairs retrieval
    accs={fp:[] for fp in LANGS}
    pair_acc=[]
    for a in LANGS:
        for b in LANGS:
            if a==b: continue
            sims=emb[a]@emb[b].T
            top1=(sims.argmax(1)==np.arange(len(idx))).mean()
            ranks=(-sims).argsort(1)
            rr=np.mean([1.0/(1+np.where(ranks[i]==i)[0][0]) for i in range(len(idx))])
            pair_acc.append(top1); accs[a].append(top1); accs[b].append(top1)
            rows.append({'model':mid,'src':disp[a],'tgt':disp[b],'top1':top1,'mrr':rr})
    print('%s  mean top1=%.3f'%(mid, np.mean(pair_acc)))
    for fp in LANGS: per_lang.append({'model':mid,'language':disp[fp],'mean_top1':np.mean(accs[fp])})
    del model, emb; torch.cuda.empty_cache() if torch.cuda.is_available() else None

res=pd.DataFrame(rows); res.to_csv('retrieval_results.csv', index=False)
pl=pd.DataFrame(per_lang); pl.to_csv('retrieval_per_language.csv', index=False)
print('saved retrieval_results.csv + retrieval_per_language.csv')
pl.sort_values('mean_top1').head(15)


## 5. Probe: meaning vs language family
Embed the same verses in every language with one model; check whether nearest neighbours of a verse are the *same verse in other languages* (meaning) or *other verses in the same language* (language). Saves `meaning_vs_language.csv`.

In [ ]:
mid=MODELS[0]
model=SentenceTransformer(mid, device='cuda' if torch.cuda.is_available() else 'cpu')
pre='query: ' if 'e5' in mid else ''
S=min(300,len(idx)); sub=idx[:S]
X=[]; meta=[]
for fp in LANGS:
    e=encode(model,[str(data[fp][k]) for k in sub], pre)
    X.append(e); meta+= [(disp[fp], k) for k in sub]
X=np.vstack(X)
sims=X@X.T; np.fill_diagonal(sims,-1)
nn=sims.argmax(1)
same_meaning=np.mean([meta[i][1]==meta[nn[i]][1] for i in range(len(meta))])
same_language=np.mean([meta[i][0]==meta[nn[i]][0] for i in range(len(meta))])
pd.DataFrame([{'model':mid,'nn_same_verse_other_lang':same_meaning,'nn_same_language_other_verse':same_language}]).to_csv('meaning_vs_language.csv', index=False)
print('NN is same verse (meaning): %.3f | NN is same language (family): %.3f'%(same_meaning,same_language))


## 6. Bundle → `quran_retrieval_results.zip`

In [ ]:
out=[f for f in ['retrieval_results.csv','retrieval_per_language.csv','meaning_vs_language.csv'] if os.path.exists(f)]
json.dump({'models':MODELS,'sample':len(idx),'languages':len(LANGS),'timestamp':time.strftime('%Y-%m-%d %H:%M')}, open('retrieval_metadata.json','w'), indent=2)
out.append('retrieval_metadata.json')
with zipfile.ZipFile('quran_retrieval_results.zip','w') as z:
    for f in out: z.write(f)
from google.colab import files; files.download('quran_retrieval_results.zip')
